In [12]:
import pandas as pd
import numpy as np
import datetime as dt



#Read the Data 


data = pd.read_excel(r"C:\Users\Eric\OneDrive - McGill University\Desktop\Python\python_training\Sales.xlsx")


In [13]:
# Inspect the first few rows of the DataFrame

data.head()
data.head()

data.info()
data.info()

data.isna().sum()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         508 non-null    object 
 1   Order_Date       468 non-null    object 
 2   Category         508 non-null    object 
 3   Region           508 non-null    object 
 4   Units_Sold       489 non-null    object 
 5   Unit_Price       478 non-null    object 
 6   Customer_Age     482 non-null    float64
 7   Customer_Rating  478 non-null    float64
dtypes: float64(2), object(6)
memory usage: 31.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         508 non-null    object 
 1   Order_Date       468 non-null    object 
 2   Category         508 non-null    object 
 3   Region           508 non-null    object 
 4 

Order_ID            0
Order_Date         40
Category            0
Region              0
Units_Sold         19
Unit_Price         30
Customer_Age       26
Customer_Rating    30
dtype: int64

In [14]:
# Data Quality check 

for columns in data.columns:
    missing_pct = data[columns].isnull().mean()*100
    if missing_pct > 0 :
        print(f"Column '{columns}' is missing {missing_pct:.1f}% of data")

# Check which rows have missing values

data[data.isnull().any(axis =1)]


#find duplicate 
data.duplicated().sum()

# Filter DataFrame to inspect duplicate rows
duplicates_data = data[data.duplicated(keep= False)]
print(duplicates_data)




Column 'Order_Date' is missing 7.9% of data
Column 'Units_Sold' is missing 3.7% of data
Column 'Unit_Price' is missing 5.9% of data
Column 'Customer_Age' is missing 5.1% of data
Column 'Customer_Rating' is missing 5.9% of data
     Order_ID  Order_Date        Category   Region Units_Sold Unit_Price  \
10   ORD-1010  2023-02-14     Electronics    North          7     $51.82   
11   ORD-1011  2023-10-30  Home & Kitchen     West         11      35.32   
12   ORD-1012  2023-08-05     Electronics       N.         11        NaN   
13   ORD-1013  01/17/2023  Home & Kitchen     EAST          8     274.91   
14   ORD-1014  2023-01-16  Home & Kitchen    North          5     322.34   
15   ORD-1015  2023-02-17  Home & Kitchen  Central          4     365.78   
16   ORD-1016  2023-04-22           Books    South          8     488.17   
17   ORD-1017  2023-04-30        Clothing       N.          8     262.99   
500  ORD-1010  2023-02-14     Electronics    North          7     $51.82   
501  ORD-1011

In [15]:
#STEP 2 : DATA CLEANING 
#-Inconsistent Date Formats: Mixed YYYY-MM-DD and MM/DD/YYYY strings.
#Inconsistent Categories & Regions: Extra whitespace, random lowercasing, and typos (e.g., 'Electrronics').
#Data Type Violations: Currency symbols ($) formatted as text, numbers written as words (e.g., 'Five').
#Missing Data (NaNs): Null entries scattered across dates, prices, units sold, ratings, and customer ages.
#Outliers & Bad Data: Negative customer ages, impossible ages (e.g., 150), and extreme price outliers.
# Outliers & Bad Data: Negative customer ages, impossible ages (e.g., 150), and extreme price outliers.
#Duplicate Rows: Contains duplicate records to identify and clean.
#




In [16]:
#-Inconsistent Date Formats: Mixed YYYY-MM-DD and MM/DD/YYYY strings.

def clean_data(data):
     # Convert dates to datetime
    data["Order_Date"] = pd.to_datetime(data["Order_Date"], errors= "coerce")
    return data 
# Pass 'data' into the function
data= clean_data(data)
# Check the updated column data type and first few rows
print(data["Order_Date"].head())

    




0   2023-11-24
1   2023-02-27
2   2023-01-13
3   2023-05-21
4          NaT
Name: Order_Date, dtype: datetime64[ns]


In [17]:
#Inconsistent Categories & Regions: Extra whitespace, random lowercasing, and typos (e.g., 'Electrronics').

# 1. Clean Category Column
# Trim extra spaces and convert to Title Case
data['Category'] = data['Category'].astype(str).str.strip().str.title()
# Fix specific typos

Category_mapping = { 
    'Electrronics' : 'Electronics',
    'Nan' : 'None'

}
data['Category'] = data['Category'].replace(Category_mapping)

# 2. Clean Region Column
# Trim extra spaces and convert to Title Case
data['Region'] = data['Region'].astype(str).str.strip().str.title()

# Fix abbreviations and casing inconsistencies
region_mapping = {
    'N.': 'North',
    'Nan': None,
    'EAST': 'east',
    'CENTRAL':'Central',
    'SOUTH' : 'South'
}
data['Region'] = data['Region'].replace(region_mapping)

In [18]:
#Data Type Violations: Currency symbols ($) formatted as text, numbers written as words (e.g., 'Five').
#Convert value to string

data['Unit_Price'] =( data['Unit_Price'].astype(str).replace('$', '', regex= False ).str.strip()
)
data['Unit_Price'] = pd.to_numeric(data['Unit_Price'], errors='coerce')

#FOR column 'Units_Sold':

word_to_num= {
'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
}

# Normalize string values to lowercase before mapping
cleaned_units= data['Units_Sold'].astype(str).str.strip().str.lower()
data['Units_Sold']= cleaned_units.replace(word_to_num)

# Convert to numeric float/integer
data['Units_Sold'] = pd.to_numeric(data['Units_Sold'], errors='coerce')



In [28]:
#Missing Data (NaNs): Null entries scattered across dates, prices, units sold, ratings, and customer ages.

def missing_data(data):

 try:
  
 #Copy the data 
    Cleaned_data =data.copy()

   #Handling missing for Date 
    if 'Order_Date' in Cleaned_data.columns:
     Cleaned_data['Order_Date'] = Cleaned_data['Order_Date'].fillna('Unknown')

    if 'Units_Sold' in Cleaned_data.columns:
      Cleaned_data['Units_Sold'] = Cleaned_data['Units_Sold'].fillna('Unknown')

    if 'Unit_Price' in Cleaned_data.columns:
      Cleaned_data['Unit_Price'] = Cleaned_data['Unit_Price'].fillna('Unknown')

    if 'Customer_Rating' in Cleaned_data.columns:  
      Cleaned_data['Customer_Rating'] = Cleaned_data['Customer_Rating'].fillna('Unknown')

    
    if 'Customer_Age' in Cleaned_data.columns:  
      Cleaned_data['Customer_Age'] = Cleaned_data['Customer_Age'].fillna('Unknown')


   
    return Cleaned_data

 except Exception as e:
        print(f"Error during missing data handling: {e}")
        return data

new_data = missing_data(data)
   

In [35]:
# Identify the duplicate data  and Clean the Data 


def remove_duplicate(new_data):

    #Count the number of the duplicated data 
    data_count = new_data.duplicated().sum()
    print(f"the number of duplicated rows is : {data_count} duplicated rows ")

    #REMOVE DUPLICATE ROWS:

    drop_data = new_data.drop_duplicates(keep= 'first')

    #RESET DATAFRAME INDEX

    reset_data = new_data.reset_index(drop=True)

    return new_data

new_version_data = remove_duplicate(new_data)


the number of duplicated rows is : 8 duplicated rows 


In [ ]:
# Analysis Question 